In [93]:
import pandas as pd

In [94]:
df_ord = pd.read_csv('../dataset1/orders.csv')
df_item = pd.read_csv('../dataset1/order_items.csv')
df_pay = pd.read_csv('../dataset1/payments.csv')
df_ship = pd.read_csv('../dataset1/shipments.csv')

df_ord.info()
df_item.info()
df_pay.info()
df_ship.info()

C:\Users\thaim\AppData\Local\Temp\ipykernel_20320\2714234306.py:2: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_item = pd.read_csv('../dataset1/order_items.csv')


<class 'pandas.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   order_id        646945 non-null  int64
 1   order_date      646945 non-null  str  
 2   customer_id     646945 non-null  int64
 3   zip             646945 non-null  int64
 4   order_status    646945 non-null  str  
 5   payment_method  646945 non-null  str  
 6   device_type     646945 non-null  str  
 7   order_source    646945 non-null  str  
dtypes: int64(3), str(5)
memory usage: 39.5 MB
<class 'pandas.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  int64  
 1   product_id       714669 non-null  int64  
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-

In [95]:
df_ord = df_ord[['order_id', 'order_status', 'order_date']]
df_item = df_item[['order_id', 'quantity', 'unit_price', 'discount_amount']]
df_pay = df_pay[['order_id', 'payment_value', 'payment_method']]
df_ship = df_ship[['order_id', 'shipping_fee']]

same_order = df_ship.duplicated(subset='order_id', keep=False)
print(same_order.value_counts())

False    566067
Name: count, dtype: int64


In [96]:
df_ord = df_ord.merge(df_ship, on='order_id', how='left')
df_ord = df_ord.merge(df_pay, on='order_id', how='left')

df_ord['shipping_fee'] = df_ord['shipping_fee'].fillna(0)
df_item['price_pay'] = (df_item['quantity'] * df_item['unit_price']) - df_item['discount_amount']
df_ord.info()
df_item.info()

<class 'pandas.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   order_id        646945 non-null  int64  
 1   order_status    646945 non-null  str    
 2   order_date      646945 non-null  str    
 3   shipping_fee    646945 non-null  float64
 4   payment_value   646945 non-null  float64
 5   payment_method  646945 non-null  str    
dtypes: float64(2), int64(1), str(3)
memory usage: 29.6 MB
<class 'pandas.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 5 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  int64  
 1   quantity         714669 non-null  int64  
 2   unit_price       714669 non-null  float64
 3   discount_amount  714669 non-null  float64
 4   price_pay        714669 non-null  float64
dtypes: float64(3), int64(2)
memory usage: 27.

In [97]:
total_price = df_item.groupby('order_id')['price_pay'].sum()
df_ord['product_price'] = df_ord['order_id'].map(total_price)
#df_ord['product_price'] = df_ord['product_price'].fillna(0)
df_ord['total'] = df_ord['product_price'] + df_ord['shipping_fee']

df_ord['product_price'] = df_ord['product_price'].round(10)
df_ord['payment_value'] = df_ord['payment_value'].round(10)
df_ord['is_right_compute_payment'] = df_ord['product_price'] == df_ord['payment_value']
 
df_ord.info()
print(df_ord['is_right_compute_payment'].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 9 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   order_id                  646945 non-null  int64  
 1   order_status              646945 non-null  str    
 2   order_date                646945 non-null  str    
 3   shipping_fee              646945 non-null  float64
 4   payment_value             646945 non-null  float64
 5   payment_method            646945 non-null  str    
 6   product_price             646945 non-null  float64
 7   total                     646945 non-null  float64
 8   is_right_compute_payment  646945 non-null  bool   
dtypes: bool(1), float64(4), int64(1), str(3)
memory usage: 40.1 MB
is_right_compute_payment
True    646945
Name: count, dtype: int64


**-> Payments = unit_price (in order_item, not product) * quantity - discount_amount**

In [98]:
df_ret = pd.read_csv('../dataset1/returns.csv')
df_sale = pd.read_csv('../dataset1/sales.csv')
df_ord = df_ord.drop(columns=['product_price', 'total', 'is_right_compute_payment'])
df_ret.info()
df_sale.info()
df_ord.info()

<class 'pandas.DataFrame'>
RangeIndex: 39939 entries, 0 to 39938
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   return_id        39939 non-null  str    
 1   order_id         39939 non-null  int64  
 2   product_id       39939 non-null  int64  
 3   return_date      39939 non-null  str    
 4   return_reason    39939 non-null  str    
 5   return_quantity  39939 non-null  int64  
 6   refund_amount    39939 non-null  float64
dtypes: float64(1), int64(3), str(3)
memory usage: 2.1 MB
<class 'pandas.DataFrame'>
RangeIndex: 3833 entries, 0 to 3832
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Date     3833 non-null   str    
 1   Revenue  3833 non-null   float64
 2   COGS     3833 non-null   float64
dtypes: float64(2), str(1)
memory usage: 90.0 KB
<class 'pandas.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 6 columns)

In [99]:
df_ret = df_ret[['return_id', 'order_id', 'return_date', 'refund_amount']]
df_ord = df_ord.merge(df_ret, on='order_id', how='left')
df_ord['refund_amount'] = df_ord['refund_amount'].fillna(0.0)

print(df_pay['payment_method'].value_counts())
print(df_ord['order_status'].value_counts())

payment_method
credit_card      356352
paypal            97018
cod               96681
apple_pay         64763
bank_transfer     32131
Name: count, dtype: int64
order_status
delivered    516716
cancelled     59462
returned      40019
shipped       13773
paid          13577
created        7275
Name: count, dtype: int64


In [100]:
bang_cheo = pd.crosstab(df_ord['payment_method'], df_ord['order_status'])
print(bang_cheo)

order_status    cancelled  created  delivered  paid  returned  shipped
payment_method                                                        
apple_pay            5190      732      52836  1331      3584     1405
bank_transfer        2535      343      26263   688      1794      697
cod                 15468     1082      67355  2127      9554     2029
credit_card         28452     4028     291073  7405     19696     7587
paypal               7817     1090      79189  2026      5391     2055


In [101]:
df_ord['receive_money']  = df_ord['payment_method'] != 'cod'
cond_cod = (df_ord['payment_method'] == 'cod') & \
           ((df_ord['order_status'] == 'delivered') | (df_ord['order_status'] == 'returned') | (df_ord['order_status'] == 'created'))
df_ord.loc[cond_cod, 'receive_money'] = True
#df_ord = df_ord[df_ord['order_status'] != 'cancelled']


sales_revenue = df_ord.groupby('order_date')['payment_value'].sum()
ship_amount = df_ord.groupby('order_date')['shipping_fee'].sum()
return_amount = df_ord.groupby('order_date')['refund_amount'].sum()
#return_amount = df_ord.groupby('return_date')['refund_amount'].sum()

print(sales_revenue)
print(ship_amount)
print(return_amount)

print(df_ord.tail(10))


order_date
2012-07-04    5130052.31
2012-07-05    2766170.52
2012-07-06    3070014.41
2012-07-07    2667930.94
2012-07-08    2360851.90
                 ...    
2022-12-27    1680442.93
2022-12-28    2758983.34
2022-12-29    2467155.53
2022-12-30    2323393.87
2022-12-31    2015982.03
Name: payment_value, Length: 3833, dtype: float64
order_date
2012-07-04    575.96
2012-07-05    353.10
2012-07-06    237.39
2012-07-07    247.92
2012-07-08    447.98
               ...  
2022-12-27    108.84
2022-12-28     36.03
2022-12-29     66.16
2022-12-30      0.00
2022-12-31      0.00
Name: shipping_fee, Length: 3833, dtype: float64
order_date
2012-07-04    171067.01
2012-07-05     57342.91
2012-07-06    195614.58
2012-07-07    134785.14
2012-07-08    135871.83
                ...    
2022-12-27         0.00
2022-12-28         0.00
2022-12-29         0.00
2022-12-30         0.00
2022-12-31         0.00
Name: refund_amount, Length: 3833, dtype: float64
        order_id order_status  order_date  shipp

In [102]:
net_revenue = sales_revenue
#net_revenue = net_revenue.add(-return_amount, fill_value=0)
#net_revenue = net_revenue.add(-ship_amount, fill_value=0)
#net_revenue = sales_revenue - return_amount
#net_revenue -= ship_amount
print(net_revenue)
df_sale['compute'] = df_sale['Date'].map(net_revenue)
df_sale['is_right_compute_revenue'] = df_sale['compute'] == df_sale['Revenue']
df_sale.info()
print(df_sale['is_right_compute_revenue'].value_counts())

df_sale['diff'] = df_sale['compute'] - df_sale['Revenue']
print(df_sale[['Date', 'compute', 'Revenue', 'diff']].head(20))

order_date
2012-07-04    5130052.31
2012-07-05    2766170.52
2012-07-06    3070014.41
2012-07-07    2667930.94
2012-07-08    2360851.90
                 ...    
2022-12-27    1680442.93
2022-12-28    2758983.34
2022-12-29    2467155.53
2022-12-30    2323393.87
2022-12-31    2015982.03
Name: payment_value, Length: 3833, dtype: float64
<class 'pandas.DataFrame'>
RangeIndex: 3833 entries, 0 to 3832
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Date                      3833 non-null   str    
 1   Revenue                   3833 non-null   float64
 2   COGS                      3833 non-null   float64
 3   compute                   3833 non-null   float64
 4   is_right_compute_revenue  3833 non-null   bool   
dtypes: bool(1), float64(3), str(1)
memory usage: 123.7 KB
is_right_compute_revenue
False    2830
True     1003
Name: count, dtype: int64
          Date     compute     Revenue       d